In [1]:
from datasets import load_dataset

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ds = load_dataset("lmarena-ai/PPE-Human-Preference-V1")

In [3]:
ds = ds['test']

In [4]:
set(ds['model_a']).union(set(ds['model_b']))

{'claude-3-haiku-20240307',
 'claude-3-opus-20240229',
 'claude-3-sonnet-20240229',
 'command-r',
 'command-r-plus',
 'gemini-1.5-flash-api-0514',
 'gemini-1.5-pro-api-0514',
 'gpt-4-0314',
 'gpt-4-0613',
 'gpt-4-1106-preview',
 'gpt-4-turbo-2024-04-09',
 'gpt-4o-2024-05-13',
 'llama-3-70b-instruct',
 'llama-3-8b-instruct',
 'mistral-large-2402',
 'phi-3-medium-4k-instruct',
 'qwen1.5-72b-chat',
 'qwen2-72b-instruct',
 'starling-lm-7b-beta',
 'yi-1.5-34b-chat'}

In [5]:
open_weight_models = [
    'qwen2-72b-instruct',
    'llama-3-8b-instruct',
    'llama-3-70b-instruct',
    'phi-3-medium-4k-instruct',
    'qwen1.5-72b-chat',
    'qwen2-72b-instruct',
    
    'mistral-large-2402',
    
]

print(f"Found {len(open_weight_models)} open-weight models")

Found 7 open-weight models


In [6]:
import pandas as pd
from collections import defaultdict

def analyze_model(model_name, dataset):
    """
    Analyze win/loss statistics for a specific model.
    Excludes ties.
    """
    # Filter to rows where this model appears in either position
    model_rows = dataset.filter(
        lambda x: x['model_a'] == model_name or x['model_b'] == model_name
    )
    
    # Exclude ties (winner == 'tie' or 'tie (bothbad)')
    non_tie_rows = model_rows.filter(
        lambda x: x['winner'] not in ['tie', 'tie (bothbad)']
    )
    
    # Count wins and losses
    wins = 0
    losses = 0
    opponent_stats = defaultdict(lambda: {'wins': 0, 'losses': 0})
    
    for row in non_tie_rows:
        if row['model_a'] == model_name:
            # Model is in position A
            opponent = row['model_b']
            if row['winner'] == 'model_a':
                wins += 1
                opponent_stats[opponent]['wins'] += 1
            else:  # winner == 'model_b'
                losses += 1
                opponent_stats[opponent]['losses'] += 1
        else:
            # Model is in position B
            opponent = row['model_a']
            if row['winner'] == 'model_b':
                wins += 1
                opponent_stats[opponent]['wins'] += 1
            else:  # winner == 'model_a'
                losses += 1
                opponent_stats[opponent]['losses'] += 1
    
    total_comparisons = wins + losses
    
    return {
        'model': model_name,
        'total_comparisons': total_comparisons,
        'wins': wins,
        'losses': losses,
        'win_rate': wins / total_comparisons if total_comparisons > 0 else 0,
        'opponent_stats': dict(opponent_stats),
        'dataset_view': non_tie_rows
    }

# Analyze all open-weight models
results = {}
for model in open_weight_models:
    results[model] = analyze_model(model, ds)
    
print("Analysis complete!")

Filter: 100%|██████████| 1716/1716 [00:00<00:00, 11691.42 examples/s]


Analysis complete!


In [7]:
# Create summary table
summary_data = []
for model_name, data in results.items():
    summary_data.append({
        'Model': model_name,
        'Total Comparisons': data['total_comparisons'],
        'Wins': data['wins'],
        'Losses': data['losses'],
        'Win Rate': f"{data['win_rate']:.2%}",
        'Num Opponents': len(data['opponent_stats'])
    })

summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.sort_values('Total Comparisons', ascending=False)
print(summary_df.to_string(index=False))

                   Model  Total Comparisons  Wins  Losses Win Rate  Num Opponents
    llama-3-70b-instruct               1145   640     505   55.90%             19
     llama-3-8b-instruct               1129   457     672   40.48%             19
      mistral-large-2402               1063   444     619   41.77%             19
        qwen1.5-72b-chat                943   352     591   37.33%             19
      qwen2-72b-instruct                850   417     433   49.06%             18
phi-3-medium-4k-instruct                816   246     570   30.15%             18


In [8]:
def print_model_details(model_name, data):
    """Print detailed statistics for a model."""
    print(f"\n{'='*80}")
    print(f"MODEL: {model_name}")
    print(f"{'='*80}")
    print(f"Total Comparisons (excluding ties): {data['total_comparisons']}")
    print(f"Total Wins: {data['wins']}")
    print(f"Total Losses: {data['losses']}")
    print(f"Win Rate: {data['win_rate']:.2%}")
    print(f"\nOpponent Breakdown ({len(data['opponent_stats'])} unique opponents):")
    print(f"{'-'*80}")
    
    # Create opponent table
    opponent_data = []
    for opponent, stats in data['opponent_stats'].items():
        total = stats['wins'] + stats['losses']
        win_rate = stats['wins'] / total if total > 0 else 0
        opponent_data.append({
            'Opponent': opponent,
            'Comparisons': total,
            'Wins': stats['wins'],
            'Losses': stats['losses'],
            'Win Rate': f"{win_rate:.1%}"
        })
    
    opponent_df = pd.DataFrame(opponent_data)
    opponent_df = opponent_df.sort_values('Comparisons', ascending=False)
    print(opponent_df.to_string(index=False))

# Print details for each model, sorted by total comparisons
for model_name in summary_df['Model']:
    print_model_details(model_name, results[model_name])


MODEL: llama-3-70b-instruct
Total Comparisons (excluding ties): 1145
Total Wins: 640
Total Losses: 505
Win Rate: 55.90%

Opponent Breakdown (19 unique opponents):
--------------------------------------------------------------------------------
                 Opponent  Comparisons  Wins  Losses Win Rate
       mistral-large-2402           75    50      25    66.7%
   claude-3-opus-20240229           73    31      42    42.5%
   gpt-4-turbo-2024-04-09           70    36      34    51.4%
       gpt-4-1106-preview           68    21      47    30.9%
        gpt-4o-2024-05-13           66    20      46    30.3%
               gpt-4-0613           65    41      24    63.1%
               gpt-4-0314           63    40      23    63.5%
 phi-3-medium-4k-instruct           62    44      18    71.0%
       qwen2-72b-instruct           62    38      24    61.3%
          yi-1.5-34b-chat           62    38      24    61.3%
      starling-lm-7b-beta           58    37      21    63.8%
           

In [9]:
# Show detailed stats for the top 3 models by comparison count
for model_name in summary_df['Model'].head(3):
    print_model_details(model_name, results[model_name])


MODEL: llama-3-70b-instruct
Total Comparisons (excluding ties): 1145
Total Wins: 640
Total Losses: 505
Win Rate: 55.90%

Opponent Breakdown (19 unique opponents):
--------------------------------------------------------------------------------
                 Opponent  Comparisons  Wins  Losses Win Rate
       mistral-large-2402           75    50      25    66.7%
   claude-3-opus-20240229           73    31      42    42.5%
   gpt-4-turbo-2024-04-09           70    36      34    51.4%
       gpt-4-1106-preview           68    21      47    30.9%
        gpt-4o-2024-05-13           66    20      46    30.3%
               gpt-4-0613           65    41      24    63.1%
               gpt-4-0314           63    40      23    63.5%
 phi-3-medium-4k-instruct           62    44      18    71.0%
       qwen2-72b-instruct           62    38      24    61.3%
          yi-1.5-34b-chat           62    38      24    61.3%
      starling-lm-7b-beta           58    37      21    63.8%
           

In [10]:
print("OPEN-WEIGHT MODELS ANALYSIS")
print("="*80)
print(f"\nTotal open-weight models found: {len(open_weight_models)}")
print(f"\nModels ranked by number of comparisons (excluding ties):\n")

for i, model_name in enumerate(summary_df['Model'].head(10), 1):
    data = results[model_name]
    print(f"{i}. {model_name}")
    print(f"   • {data['total_comparisons']:,} comparisons")
    print(f"   • {data['wins']:,} wins, {data['losses']:,} losses ({data['win_rate']:.1%} win rate)")
    print(f"   • Competed against {len(data['opponent_stats'])} different models")
    
    # Show top 3 most frequent opponents
    top_opponents = sorted(data['opponent_stats'].items(), 
                          key=lambda x: x[1]['wins'] + x[1]['losses'], 
                          reverse=True)[:3]
    print(f"   • Top opponents: {', '.join([opp[0] for opp in top_opponents])}")
    print()

print("\nAll dataset views are stored in the 'results' dictionary.")
print("Access them with: results['model-name']['dataset_view']")

OPEN-WEIGHT MODELS ANALYSIS

Total open-weight models found: 7

Models ranked by number of comparisons (excluding ties):

1. llama-3-70b-instruct
   • 1,145 comparisons
   • 640 wins, 505 losses (55.9% win rate)
   • Competed against 19 different models
   • Top opponents: mistral-large-2402, claude-3-opus-20240229, gpt-4-turbo-2024-04-09

2. llama-3-8b-instruct
   • 1,129 comparisons
   • 457 wins, 672 losses (40.5% win rate)
   • Competed against 19 different models
   • Top opponents: gpt-4-0613, gpt-4-1106-preview, claude-3-opus-20240229

3. mistral-large-2402
   • 1,063 comparisons
   • 444 wins, 619 losses (41.8% win rate)
   • Competed against 19 different models
   • Top opponents: llama-3-70b-instruct, claude-3-opus-20240229, gpt-4-1106-preview

4. qwen1.5-72b-chat
   • 943 comparisons
   • 352 wins, 591 losses (37.3% win rate)
   • Competed against 19 different models
   • Top opponents: gpt-4-1106-preview, claude-3-sonnet-20240229, gpt-4-turbo-2024-04-09

5. qwen2-72b-instru

In [13]:
import json
from pathlib import Path

def extract_assistant_response(conversation):
    """Extract the assistant's response from a conversation list."""
    return {"role": "assistant","content": conversation}

def extract_user_prompt(conversation):
    """Extract the user prompt from a conversation list."""
    
        
        # Get the text from the first content item
    
    return {"role": "user","content": conversation}

def export_model_to_jsonl(model_name, model_data, output_dir):
    """
    Export a model's Arena data to JSONL format for self-preference testing.
    
    For Arena data, we'll use:
    - question: the user prompt
    - judge_completion: the model's own response (when it won or lost)
    - ref_completion: the opponent's response
    - judge_correct: 1 if model won, 0 if model lost
    - ref_correct: 0 if model won, 1 if model lost
    """
    output_path = output_dir / f"{model_name.replace('/', '_')}_arena.jsonl"
    
    dataset = model_data['dataset_view']
    opponent_stats = model_data['opponent_stats']
    
    exported_count = 0
    with open(output_path, 'w', encoding='utf-8') as f:
        for row in dataset:
            # Extract prompt from conversation
            prompt = extract_user_prompt(row['prompt'])
            
            # Determine if the model is in position A or B
            if row['model_a'] == model_name:
                judge_response = extract_assistant_response(row['response_1'])
                ref_response = extract_assistant_response(row['response_2'])
                opponent = row['model_b']
                judge_won = 1 if row['winner'] == 'model_a' else 0
            else:  # model is in position B
                judge_response = extract_assistant_response(row['response_2'])
                ref_response = extract_assistant_response(row['response_1'])
                opponent = row['model_a']
                judge_won = 1 if row['winner'] == 'model_b' else 0
            
            record = {
                'question': prompt,
                'judge_completion': judge_response,
                'ref_completion': ref_response,
                'judge_correct': judge_won,
                'ref_correct': 1 - judge_won,
                'opponent': opponent,
                'arena_id': row.get('id', ''),
                'category_tag': row.get('category_tag', ''),
                'language': row.get('language', ''),
            }
            
            f.write(json.dumps(record, ensure_ascii=False) + '\n')
            exported_count += 1
    
    return exported_count, output_path

# Create output directory
output_dir = Path('arena_diffs')
output_dir.mkdir(exist_ok=True)

# Export data for each model
export_summary = []
for model_name in open_weight_models:
    if model_name in results:
        count, path = export_model_to_jsonl(model_name, results[model_name], output_dir)
        export_summary.append({
            'model': model_name,
            'examples': count,
            'file': str(path)
        })
        print(f"✓ Exported {count} examples for {model_name}")

print(f"\n{'='*80}")
print(f"Exported {len(export_summary)} model datasets to {output_dir}/")
print(f"{'='*80}")

✓ Exported 850 examples for qwen2-72b-instruct
✓ Exported 1129 examples for llama-3-8b-instruct
✓ Exported 1145 examples for llama-3-70b-instruct
✓ Exported 816 examples for phi-3-medium-4k-instruct
✓ Exported 943 examples for qwen1.5-72b-chat
✓ Exported 850 examples for qwen2-72b-instruct
✓ Exported 1063 examples for mistral-large-2402

Exported 7 model datasets to arena_diffs/
